# Notebook 09: Final Comparison & Experimental Analysis
This flagship notebook runs the complete comparative benchmark across all solvers, generates high-res publication figures, and performs statistical bootstrap confidence interval analysis.

In [ ]:
import sys
import os
import zipfile

# Auto-extract zip file if Kaggle uploaded dataset as a raw zip
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".zip") and "wordle" in file.lower():
                zip_path = os.path.join(root_dir, file)
                print(f"Extracting uploaded Kaggle dataset: {zip_path} -> /kaggle/working/")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/kaggle/working/")

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle",
    "/kaggle/input/wordle/wordle",
    "/kaggle/input/wordle/wordle-ai",
    "/kaggle/input/wordle-ai",
]

# Search recursively inside /kaggle/input for any folder containing 'src'
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs:
            possible_roots.insert(0, root_dir)

for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        print(f"Found 'src' package at: {abs_root}")
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")

from src.wordle.dictionary import WordleDictionary
from src.solvers.random_solver import RandomSolver
from src.solvers.frequency_solver import FrequencySolver
from src.solvers.entropy_solver import EntropySolver
from src.solvers.character_transformer_solver import CharacterTransformerSolver
from src.solvers.hybrid_solver import HybridSolver
from src.evaluation.benchmark import BenchmarkRunner
from src.evaluation.visualization import plot_benchmark_summary
from src.evaluation.statistical_tests import compute_bootstrap_ci

dict_mgr = WordleDictionary(seed=42)
runner = BenchmarkRunner(dict_mgr)
test_secrets = dict_mgr.test_secrets[:30]

solvers = [
    RandomSolver(dict_mgr, seed=42),
    FrequencySolver(dict_mgr),
    EntropySolver(dict_mgr),
    CharacterTransformerSolver(dict_mgr),
    HybridSolver(dict_mgr),
]

metrics_list = []
for s in solvers:
    m = runner.evaluate_solver(s, test_secrets=test_secrets)
    metrics_list.append(m)
    print(f"{m.model_name:<30} | Win Rate: {m.win_rate:>5.1f}% | Avg Guesses: {m.avg_guesses:>4.2f}")

plot_benchmark_summary(metrics_list, output_dir="../reports/figures/")
print("\nGenerated comparative benchmark plots saved to reports/figures/")